In [ ]:
%%capture
import os
from django_pandas.io import read_frame
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)

In [ ]:
from edc_pdutils.dataframes import get_subject_visit
from edc_pdutils.dataframes import get_crf
df_main = pd.read_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "df_main_1858.csv")
df_visit = get_subject_visit("intecomm_subject.subjectvisit")
df_main = df_main.merge(df_visit[df_visit.visit_code==1000.0][["subject_identifier", "baseline_datetime", "last_visit_datetime"]], on="subject_identifier", how="left")
df_main.reset_index(inplace=True, drop=True)

In [ ]:
# DM -- GLUCOSE 475/480
subject_identifiers = list(df_main[(df_main.dm==1) & (df_main.hiv==0)].subject_identifier)
dm_initial_review = get_crf(model="intecomm_subject.dminitialreview", subject_visit_model="intecomm_subject.subjectvisit", subject_identifiers=subject_identifiers)
df_main = df_main.merge(dm_initial_review[dm_initial_review.visit_code==1000.0][["subject_identifier", "glucose_value", "glucose_units", "glucose_date", "glucose_fasting", "glucose_fasting_duration_delta"]], on="subject_identifier", how="left")
df_main.reset_index(inplace=True, drop=True)

In [ ]:
# 5 missing baseline GLU
df_main[(df_main["dm"]==1) & (df_main["hiv"]==0) & (df_main.glucose_value.isna())][["subject_identifier", "site_id"]]


In [ ]:
# 5 missing baseline GLU
df_main[(df_main["dm"] == 1) & (df_main["hiv"] == 0) & (df_main.glucose_value.isna())][
    ["subject_identifier", "site_id"]]

In [ ]:
# was glucose withon 6m of baseline
df_main[(df_main["dm"] == 1) & (df_main["hiv"] == 0) & (df_main.glucose_value.notna()) & (
            df_main.glucose_date - df_main.baseline_datetime < pd.Timedelta(days=182))][
    ["subject_identifier", "site_id"]]

In [ ]:
# are glucose dates within 6 months of baseline (and not more than 1 month future
df_tmp = df_main.copy()
df_tmp["tdelta"] = df_main.glucose_date - df_main.baseline_datetime
df_tmp["neg182"] = pd.Timedelta(days=-182)
# df_main[(pd.Timedelta(days=-182) < df_main.glucose_date - df_main.baseline_datetime) ][["baseline_datetime", "glucose_date", df_main.glucose_date - df_main.baseline_datetime, pd.Timedelta(days=1)]]


In [ ]:
# OK, from -105 to +30
# mean 0, sd 10
df_tmp[["tdelta", "baseline_datetime", "glucose_date"]].tdelta.describe()

In [ ]:
# original fasting YES/NO == 454/21
df_main.glucose_fasting.value_counts()

In [ ]:
# reclassify as non-fasting if duration less than 8hrs
from edc_constants.constants import YES, NO, NOT_APPLICABLE
import numpy as np

def fasting_if_eight_hours(s):
    """set fasting to NO if duration less than 8hrs"""
    if s.dm == 1 and not pd.isna(s.glucose_value):
        if s.glucose_fasting_duration_delta >= pd.Timedelta(hours=8):
            return YES
        return NO
    return np.nan

df_main["glucose_fasting_orig"] = df_main["glucose_fasting"]
df_main["glucose_fasting"] = df_main.apply(fasting_if_eight_hours, axis=1)
df_main.glucose_fasting.value_counts()

In [ ]:
# all glucose values for those screened with DM
df_main[df_main.glucose_value.notna()]["glucose_value"].count()

In [ ]:
# describe fasting 411/475
df_main[df_main.glucose_fasting==YES].glucose_value.describe()

In [ ]:
# describe non-fasting 64/475
df_main[df_main.glucose_fasting==NO].glucose_value.describe()

In [ ]:
# a fasting glucose lower than 4 should be queried?
df_main[(df_main.hiv==0) & (df_main.dm==1) & (df_main.glucose_fasting==YES) & (df_main.glucose_value < 4.0)]

In [ ]:
not_dm = ["107-205-0017-1"]
mssing_baseliner_glu = []

In [ ]:
from edc_constants.constants import NOT_APPLICABLE
dm_initial_review[(dm_initial_review.glucose_units==NOT_APPLICABLE)]
# dm_initial_review.glucose_units.value_counts()

In [ ]:
df_visit = get_subject_visit("intecomm_subject.subjectvisit")

In [ ]:
from edc_appointment.analytics import get_appointment_df
df_appointment = get_appointment_df()

In [ ]:
from edc_appointment.constants import MISSED_APPT
df_appointment[df_appointment.appt_timing==MISSED_APPT].appt_status.value_counts()

In [ ]:
from intecomm_subject.models import SubjectVisitMissed
df_visit_missed = read_frame(SubjectVisitMissed.objects.all(), verbose=False)

In [ ]:
df_visit_missed.count()

In [ ]:
df_visit_missed.columns

In [ ]:
df_appointment.appt_type.value_counts()